In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('data_football_ratings.csv')

In [4]:
df = df[df["rater"]== 'WhoScored']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 1 to 50651
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             21354 non-null  object 
 1   date                    21354 non-null  object 
 2   match                   21354 non-null  object 
 3   team                    21354 non-null  object 
 4   pos                     21354 non-null  object 
 5   pos_role                21354 non-null  object 
 6   player                  21354 non-null  object 
 7   rater                   21354 non-null  object 
 8   is_human                21354 non-null  int64  
 9   original_rating         21354 non-null  float64
 10  goals                   21354 non-null  int64  
 11  assists                 21354 non-null  int64  
 12  shots_ontarget          21354 non-null  int64  
 13  shots_offtarget         21354 non-null  int64  
 14  shotsblocked            21354 non-null  int

In [5]:
cols_to_drop = [
    "competition", "date", "match", "team", "player", "rater", "is_human",
    "degree_centrality", "betweenness_centrality", "closeness_centrality",
    "flow_centrality", "flow_success", "betweenness2goals","pos_role"
]


In [6]:
df = df.drop(columns=cols_to_drop)

In [7]:
df = df[df['pos'] != 'GK'].copy()



In [8]:
vars_portero = [
    'goals_ag_otb', 'goals_ag_itb',
    'saves_itb', 'saves_otb', 'saved_pen'
]
df = df.drop(columns=vars_portero)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19783 entries, 1 to 50651
Data columns (total 44 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pos               19783 non-null  object 
 1   original_rating   19783 non-null  float64
 2   goals             19783 non-null  int64  
 3   assists           19783 non-null  int64  
 4   shots_ontarget    19783 non-null  int64  
 5   shots_offtarget   19783 non-null  int64  
 6   shotsblocked      19783 non-null  int64  
 7   chances2score     19783 non-null  int64  
 8   drib_success      19783 non-null  int64  
 9   drib_unsuccess    19783 non-null  int64  
 10  keypasses         19783 non-null  int64  
 11  touches           19783 non-null  int64  
 12  passes_acc        19783 non-null  int64  
 13  passes_inacc      19783 non-null  int64  
 14  crosses_acc       19783 non-null  int64  
 15  crosses_inacc     19783 non-null  int64  
 16  lballs_acc        19783 non-null  int64  
 17

In [10]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Separar X y y
X = df.drop(columns=["original_rating"])
y = df["original_rating"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Suponiendo que tus categóricas son del tipo 'object' o 'category' en pandas
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Para CatBoost se usa índice, para LightGBM y XGBoost nombres (o índices)
cat_features_idx = [X.columns.get_loc(col) for col in cat_features]


In [14]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.7 MB/s eta 0:00:00


In [12]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

# Entrenar modelo
catboost = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost.fit(X_train, y_train, cat_features=cat_features_idx)

# Predecir en test
y_pred = catboost.predict(X_test)

# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7593
CatBoost RMSE: 0.3443


In [16]:
import pandas as pd

# Obtener importancias de las características
importancias = catboost.get_feature_importance(prettified=True)

print(importancias)

# Opcional: mostrar solo las 10 variables más importantes
print(importancias.head(10))


          Feature Id  Importances
0              goals    25.398049
1            touches    17.036716
2               lost     7.912483
3            assists     7.835214
4                win     7.141637
5          aerials_w     6.507423
6          grduels_w     3.798065
7     shots_ontarget     3.490667
8       drib_success     2.552698
9         clearances     2.220034
10         keypasses     2.166611
11           tackles     2.103001
12     interceptions     1.434712
13        stop_shots     0.975705
14            ycards     0.874575
15        passes_acc     0.767153
16               pos     0.707505
17         poss_lost     0.582934
18      dangmistakes     0.572989
19       crosses_acc     0.556729
20        lballs_acc     0.524081
21         aerials_l     0.522797
22            rcards     0.493190
23     minutesPlayed     0.383489
24     crosses_inacc     0.365705
25      passes_inacc     0.333983
26       countattack     0.320885
27             fouls     0.306692
28         grd

In [ ]:
# Seleccionar solo columnas numéricas
df_numericas = df.select_dtypes(include=['number'])

# Calcular correlación entre variables numéricas
correlation = df_numericas.corr()

# Correlación de las variables numéricas con la variable objetivo 'original_rating'
corr_target = correlation['original_rating'].sort_values(ascending=False)

print(corr_target)


original_rating     1.000000
goals               0.519952
touches             0.437886
shots_ontarget      0.429117
win                 0.393060
grduels_w           0.386737
poss_lost           0.361844
assists             0.333099
minutesPlayed       0.331980
passes_acc          0.323619
passes_inacc        0.291404
aerials_w           0.273616
drib_success        0.267642
chances2score       0.248318
countattack         0.248297
tackles             0.246746
keypasses           0.234353
interceptions       0.227904
lballs_acc          0.219024
wasfouled           0.215547
tballs_acc          0.199745
shots_offtarget     0.186203
lballs_inacc        0.185837
clearances          0.185438
crosses_acc         0.185424
grduels_l           0.177257
tballs_inacc        0.154996
shotsblocked        0.122441
aerials_l           0.113639
crosses_inacc       0.111675
is_home_team        0.109056
stop_shots          0.106509
drib_unsuccess      0.093627
fouls               0.072920
offsides      

In [13]:
import numpy as np

# Asumiendo que tienes win (1/0) y lost (1/0)
# Empate donde ni win ni lost son 1
df['result'] = np.where(df['win'] == 1, 'victory',
                 np.where(df['lost'] == 1, 'defeat', 'draw'))

# Luego elimina las columnas originales
df = df.drop(columns=['win', 'lost'])

In [14]:
df['shots_offtarget_blocked'] = df['shots_offtarget'] + df['shotsblocked']
df = df.drop(columns=['shots_offtarget', 'shotsblocked'])


In [15]:
df = df.drop(columns=['game_duration', 'tballs_acc', 'tballs_inacc'])


In [16]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Separar X y y
X = df.drop(columns=["original_rating"])
y = df["original_rating"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Suponiendo que tus categóricas son del tipo 'object' o 'category' en pandas
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Para CatBoost se usa índice, para LightGBM y XGBoost nombres (o índices)
cat_features_idx = [X.columns.get_loc(col) for col in cat_features]


In [18]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

# Entrenar modelo
catboost = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=4, random_seed=42, verbose=0)
catboost.fit(X_train, y_train, cat_features=cat_features_idx)

# Predecir en test
import numpy as np

y_pred = catboost.predict(X_test)
y_pred = np.clip(y_pred, 0, 10)  # Fuerza el rango [0,10]


# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7601
CatBoost RMSE: 0.3438


In [ ]:
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor

# Definir modelo base
catboost = CatBoostRegressor(random_seed=42, verbose=0)

# Definir rejilla de hiperparámetros
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [300, 500],  # mantenemos bajo para que no tarde demasiado
    'l2_leaf_reg': [1, 3, 5]
}

# Configurar GridSearchCV
grid = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring='r2',     # optimizamos R²
    cv=3,             # validación cruzada 3-fold (rápida)
    n_jobs=-1
)

# Ejecutar búsqueda
grid.fit(X_train, y_train, cat_features=cat_features_idx)

print("Mejores parámetros encontrados:")
print(grid.best_params_)
print(f"Mejor R² en validación: {grid.best_score_:.4f}")

# Reentrenar modelo con mejores parámetros
best_catboost = grid.best_estimator_
y_pred = best_catboost.predict(X_test)

# Evaluar en test
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² en test: {r2:.4f}")
print(f"RMSE en test: {rmse:.4f}")


Mejores parámetros encontrados:
{'depth': 4, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
Mejor R² en validación: 0.7581
R² en test: 0.7601
RMSE en test: 0.3438


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19783 entries, 1 to 50651
Data columns (total 39 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   pos                      19783 non-null  object 
 1   original_rating          19783 non-null  float64
 2   goals                    19783 non-null  int64  
 3   assists                  19783 non-null  int64  
 4   shots_ontarget           19783 non-null  int64  
 5   chances2score            19783 non-null  int64  
 6   drib_success             19783 non-null  int64  
 7   drib_unsuccess           19783 non-null  int64  
 8   keypasses                19783 non-null  int64  
 9   touches                  19783 non-null  int64  
 10  passes_acc               19783 non-null  int64  
 11  passes_inacc             19783 non-null  int64  
 12  crosses_acc              19783 non-null  int64  
 13  crosses_inacc            19783 non-null  int64  
 14  lballs_acc               19

In [ ]:
df.head()

,pos,original_rating,goals,assists,shots_ontarget,chances2score,drib_success,drib_unsuccess,keypasses,touches,...,rcards,dangmistakes,countattack,offsides,missed_penalties,owngoals,is_home_team,minutesPlayed,result,shots_offtarget_blocked
1,DF,6.56,0,0,0,0,0,0,0,34,...,0,0,2,1,0,0,0,90,defeat,0
3,Sub,6.19,0,0,1,1,1,0,0,10,...,0,0,0,0,0,0,1,13,victory,0
5,MF,6.58,0,0,0,1,2,0,0,60,...,0,0,2,0,0,0,0,90,defeat,1
8,FW,7.34,1,0,2,3,0,0,0,47,...,0,0,1,3,0,0,1,90,victory,2
11,DF,6.38,0,0,0,0,0,0,0,80,...,0,0,1,0,0,0,0,90,defeat,2


In [ ]:
df.columns

Index(['pos', 'original_rating', 'goals', 'assists', 'shots_ontarget',
       'chances2score', 'drib_success', 'drib_unsuccess', 'keypasses',
       'touches', 'passes_acc', 'passes_inacc', 'crosses_acc', 'crosses_inacc',
       'lballs_acc', 'lballs_inacc', 'grduels_w', 'grduels_l', 'aerials_w',
       'aerials_l', 'poss_lost', 'fouls', 'wasfouled', 'clearances',
       'stop_shots', 'interceptions', 'tackles', 'dribbled_past', 'ycards',
       'rcards', 'dangmistakes', 'countattack', 'offsides', 'missed_penalties',
       'owngoals', 'is_home_team', 'minutesPlayed', 'result',
       'shots_offtarget_blocked'],
      dtype='object')

In [19]:
import pandas as pd

# Definir un caso sintético con las variables listadas
caso_sintetico = pd.DataFrame([
    {
        'pos': 'FW',
        'goals': 0,
        'assists': 0,
        'shots_ontarget': 1,
        'chances2score': 3,
        'drib_success': 4,
        'drib_unsuccess': 1,
        'keypasses': 3,
        'touches': 30,
        'passes_acc': 25,
        'passes_inacc': 5,
        'crosses_acc': 1,
        'crosses_inacc': 2,
        'lballs_acc': 4,
        'lballs_inacc': 1,
        'grduels_w': 3,
        'grduels_l': 1,
        'aerials_w': 2,
        'aerials_l': 1,
        'poss_lost': 15,
        'fouls': 1,
        'wasfouled': 2,
        'clearances': 0,
        'stop_shots': 0,
        'interceptions': 1,
        'tackles': 1,
        'dribbled_past': 1,
        'ycards': 1,
        'rcards': 0,
        'dangmistakes': 1,
        'countattack': 0,
        'offsides': 1,
        'missed_penalties': 1,
        'owngoals': 0,
        'is_home_team': 0,
        'minutesPlayed': 90,
        'result': 'victory',
        'shots_offtarget_blocked': 2
    }
])




In [20]:
def predecir_limitado(model, X, min_val=0, max_val=10):
    y_pred = model.predict(X)
    return np.clip(y_pred, min_val, max_val)


In [21]:
predecir_limitado(catboost,caso_sintetico )

array([6.93814676])

In [22]:
catboost.save_model("catboost_avanzado.cbm")
